# Bone marrow cell detection with YOLO11n

This notebook trains and evaluates a single-class bounding-box detector on DICOM whole-slide images of bone marrow aspirate smears. Consensus annotations provide the reference boxes. All annotated cell types map to the `cell` class. Annotations labeled `artifact` are excluded, while technically unfit and ambiguously typed cells are retained as localization targets.

The notebook covers three workflows.

1. **Single-patient baseline.** Train on one annotated monolayer region and validate on another region of the same slide.
2. **Patient-level cross-validation.** Train a fresh model in each of two folds and evaluate it on the patient held out from training.
3. **Visual inspection.** Apply the saved baseline checkpoint to a full-resolution crop from the second slide.

Use the repository's Python environment as the notebook kernel. See [repository setup](../README.md#setup) for the base dependencies. If Ultralytics is missing, install it in the selected kernel with `%pip install "ultralytics>=8.3,<9"`. Training uses a local YOLO11n checkpoint when available and otherwise downloads the pretrained weights. Outputs are written under `_data/yolo_cell_boxes/`.

[Ultralytics detection dataset format](https://docs.ultralytics.com/datasets/detect) | [YOLO11 training API](https://docs.ultralytics.com/models/yolo11)

## Setup

Run this cell after starting or restarting the kernel. It resolves the repository root, loads the dependencies, and defines the input paths and training settings. The expected layout is `Data/<patient_id>/MS/` for the DICOM pyramid and `Data/<patient_id>/ANN/` for the consensus boxes and annotated region outlines. Pyramid level 0 is the full-resolution image.

In [ ]:
import csv
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pydicom
import yaml
from IPython.display import display
from PIL import Image, ImageDraw
from wsidicom import WsiDicom

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from annotation_classification.geometry_serialization import (  # noqa: E402
    load_dicom_annotation_rectangles,
)

PATIENT = "A6BBC91AE73DD21C0533F735470A9CD0"
DATA_DIR = ROOT / "Data" / PATIENT
WSI_DIR = DATA_DIR / "MS"
ANN_DIR = DATA_DIR / "ANN"
OUTPUT_DIR = ROOT / "_data" / "yolo_cell_boxes"

TILE_SIZE = 1024
TRAIN_STRIDE = 512
VAL_REGION_INDEX = 1  # Zero-based index in the ROI DICOM file
local_weights = OUTPUT_DIR / "weights" / "yolo11n.pt"
MODEL_WEIGHTS = str(local_weights) if local_weights.is_file() else "yolo11n.pt"
EPOCHS = 50
BATCH_SIZE = 4
SEED = 42

## Prepare the single-patient dataset

Load the first patient's consensus rectangles and monolayer region outlines. Verify that both annotation files reference the full-resolution slide and that the two regions are in bounds and do not overlap. Print the region bounds in full-resolution slide pixels and assign one region to training and one to validation.

In [ ]:
if "ANN_DIR" not in globals():
    raise RuntimeError("Run the Setup cell above before preparing the dataset.")


def find_annotation(pattern):
    paths = sorted(ANN_DIR.glob(pattern))
    if len(paths) != 1:
        raise ValueError(
            f"Expected one matching ANN file in {ANN_DIR} for {pattern}; found {len(paths)}."
        )
    return paths[0]


def read_boxes(path, excluded_labels=()):
    annotations = load_dicom_annotation_rectangles(path)
    vertices = np.stack(
        [
            rectangle
            for rectangle, label in zip(
                annotations.rectangles_xy, annotations.labels, strict=True
            )
            if label not in excluded_labels
        ]
    )
    boxes = np.column_stack((vertices.min(axis=1), vertices.max(axis=1)))
    if not np.isfinite(boxes).all() or np.any(boxes[:, 2:] <= boxes[:, :2]):
        raise ValueError(f"Invalid rectangles in {path.name}.")
    return boxes


def labels_for_tile(boxes, x, y, size):
    origin = np.array([x, y])
    start = np.clip(boxes[:, :2] - origin, 0, size)
    end = np.clip(boxes[:, 2:] - origin, 0, size)
    visible = (end > start).all(axis=1)
    start, end = start[visible], end[visible]
    return np.column_stack(((start + end) / 2, end - start)) / size


def tile_starts(start, stop, stride):
    last = stop - TILE_SIZE
    return sorted({*range(start, last + 1, stride), last})


consensus_path = find_annotation("*consensus*.dcm")
roi_path = find_annotation("*Monolayer*.dcm")
image_paths = sorted(WSI_DIR.glob("*.dcm"))
if not image_paths:
    raise FileNotFoundError(f"No slide DICOMs in {WSI_DIR}")

image_headers = [pydicom.dcmread(path, stop_before_pixels=True) for path in image_paths]
volume_headers = [
    header for header in image_headers if hasattr(header, "TotalPixelMatrixColumns")
]
base = max(
    volume_headers,
    key=lambda hdr: hdr.TotalPixelMatrixColumns * hdr.TotalPixelMatrixRows,
)

for path in (consensus_path, roi_path):
    header = pydicom.dcmread(path, stop_before_pixels=True)
    refs = {
        item.ReferencedSOPInstanceUID
        for item in header.get("ReferencedImageSequence", [])
    }
    if header.get("PixelOriginInterpretation") != "VOLUME" or refs != {
        base.SOPInstanceUID
    }:
        raise ValueError(
            f"{path.name} does not reference the level-0 slide pixel matrix."
        )

boxes = read_boxes(consensus_path, excluded_labels=("artifact",))
regions = read_boxes(roi_path)
if len(regions) != 2 or VAL_REGION_INDEX not in range(len(regions)):
    raise ValueError(
        "This notebook expects two regions and a valid validation-region index."
    )
if np.all(
    np.minimum(regions[0, 2:], regions[1, 2:])
    > np.maximum(regions[0, :2], regions[1, :2])
):
    raise ValueError("Training and validation regions overlap.")

regions = np.column_stack((np.ceil(regions[:, :2]), np.floor(regions[:, 2:]))).astype(
    int
)
slide_size = np.array([base.TotalPixelMatrixColumns, base.TotalPixelMatrixRows])
if np.any(regions[:, :2] < 0) or np.any(regions[:, 2:] > slide_size):
    raise ValueError("An annotation region extends outside the slide.")
sizes = regions[:, 2:] - regions[:, :2]
if (
    TILE_SIZE <= 0
    or TILE_SIZE % 32
    or not 0 < TRAIN_STRIDE <= TILE_SIZE
    or np.any(sizes < TILE_SIZE)
    or np.any(sizes[VAL_REGION_INDEX] % TILE_SIZE)
):
    raise ValueError(
        "Choose a tile size divisible by 32 that tiles the validation region exactly, and a valid training stride."
    )

print(f"{len(boxes)} consensus boxes. Region bounds (x0, y0, x1, y1):")
for index, bounds in enumerate(regions):
    print(index, "val" if index == VAL_REGION_INDEX else "train", bounds)

### Export images and labels

Read 1024 by 1024 pixel tiles at level 0. Training tiles overlap, while validation tiles cover their region without overlap. Clip boxes to the visible tile area and encode them as normalized YOLO center coordinates, widths, and heights. Write `data.yaml` and a manifest that records each tile's full-resolution slide origin.

In [ ]:
dataset_dir = OUTPUT_DIR / "datasets" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
manifest = []
with WsiDicom.open(image_paths) as slide:
    for region_index, (left, top, right, bottom) in enumerate(regions):
        split = "val" if region_index == VAL_REGION_INDEX else "train"
        stride = TILE_SIZE if split == "val" else TRAIN_STRIDE
        images = dataset_dir / "images" / split
        labels = dataset_dir / "labels" / split
        images.mkdir(parents=True)
        labels.mkdir(parents=True)

        for y in tile_starts(top, bottom, stride):
            for x in tile_starts(left, right, stride):
                stem = f"roi{region_index}_x{x}_y{y}"
                tile_labels = labels_for_tile(boxes, x, y, TILE_SIZE)
                image = slide.read_region((x, y), 0, (TILE_SIZE, TILE_SIZE)).convert(
                    "RGB"
                )
                image_path = images / f"{stem}.png"
                image.save(image_path)
                text = "".join(
                    "0 " + " ".join(f"{value:.8f}" for value in row) + "\n"
                    for row in tile_labels
                )
                label_path = labels / f"{stem}.txt"
                label_path.write_text(text, encoding="utf-8")
                manifest.append(
                    {
                        "split": split,
                        "region_index": region_index,
                        "x_level0": x,
                        "y_level0": y,
                        "tile_size": TILE_SIZE,
                        "image": image_path.relative_to(dataset_dir).as_posix(),
                        "label": label_path.relative_to(dataset_dir).as_posix(),
                        "box_count": len(tile_labels),
                    }
                )

for split in ("train", "val"):
    records = [record for record in manifest if record["split"] == split]
    count = sum(record["box_count"] for record in records)
    if not count:
        raise ValueError(f"The {split} split has no visible consensus boxes.")
    print(f"{split}: {len(records)} tiles, {count} box instances")

with (dataset_dir / "manifest.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(manifest[0]))
    writer.writeheader()
    writer.writerows(manifest)

dataset_yaml = dataset_dir / "data.yaml"
dataset_yaml.write_text(
    yaml.safe_dump(
        {
            "path": dataset_dir.as_posix(),
            "train": "images/train",
            "val": "images/val",
            "names": {0: "cell"},
        }
    ),
    encoding="utf-8",
)
print(dataset_yaml)

### Inspect exported annotations

Display one labeled training tile and one labeled validation tile. The green rectangles represent the exported reference boxes. Use this preview to verify that the crop locations and label coordinates are aligned.

In [ ]:
def annotated_preview(record):
    image = Image.open(dataset_dir / record["image"]).convert("RGB")
    draw = ImageDraw.Draw(image)
    for line in (
        (dataset_dir / record["label"]).read_text(encoding="utf-8").splitlines()
    ):
        _, xc, yc, w, h = map(float, line.split())
        draw.rectangle(
            (
                (xc - w / 2) * image.width,
                (yc - h / 2) * image.height,
                (xc + w / 2) * image.width,
                (yc + h / 2) * image.height,
            ),
            outline="lime",
            width=3,
        )
    image.thumbnail((600, 600))
    return image


preview_records = {}
for split in ("train", "val"):
    record = max(
        (record for record in manifest if record["split"] == split),
        key=lambda record: record["box_count"],
    )
    preview_records[split] = record
    print(record["image"])
    display(annotated_preview(record))

## Train the single-patient baseline

Fine-tune YOLO11n on the exported training tiles. Ultralytics evaluates the model on the held-out region and saves the highest-scoring checkpoint as `best.pt`. Set `EPOCHS = 1` in Setup for a workflow check.

In [ ]:
import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
print(f"Training device: {device}")
model = YOLO(MODEL_WEIGHTS)
model.train(
    data=str(dataset_yaml),
    imgsz=TILE_SIZE,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    device=device,
    workers=0,
    seed=SEED,
    mosaic=0.0,
    amp=False,
    project=str(OUTPUT_DIR / "runs"),
    name="cell_boxes",
)
best_weights = Path(model.trainer.best)
print(best_weights)

### Evaluate the baseline checkpoint

Reload `best.pt` and report average precision at intersection-over-union (IoU) thresholds of 0.50 and 0.50 through 0.95. Display the consensus boxes and model predictions side by side. These scores describe performance on a separate region of the same slide.

In [ ]:
detector = YOLO(str(best_weights))
metrics = detector.val(
    data=str(dataset_yaml),
    imgsz=TILE_SIZE,
    batch=BATCH_SIZE,
    device=device,
    workers=0,
    project=str(OUTPUT_DIR / "runs"),
    name="validation",
)
print(f"mAP50: {metrics.box.map50:.3f}   mAP50-95: {metrics.box.map:.3f}")

record = preview_records["val"]
prediction = detector.predict(
    source=str(dataset_dir / record["image"]),
    imgsz=TILE_SIZE,
    device=device,
    conf=0.25,
    verbose=False,
)[0]
truth_preview = annotated_preview(record)
prediction_preview = Image.fromarray(prediction.plot(pil=False)[:, :, ::-1])
prediction_preview.thumbnail((600, 600))
comparison = Image.new(
    "RGB", (truth_preview.width + prediction_preview.width, truth_preview.height)
)
comparison.paste(truth_preview, (0, 0))
comparison.paste(prediction_preview, (truth_preview.width, 0))
comparison.save(best_weights.parent.parent / "validation_example.png")
print("Consensus boxes (left) and predicted boxes (right)")
display(comparison)

## Prepare patient-level cross-validation

Run **Setup** and the first code cell under **Prepare the single-patient dataset** before this section. This cell reuses their annotation and tiling functions. It exports both annotated regions for each patient and prepares two folds, each with all training tiles from one patient and all evaluation tiles from the other.

In [ ]:
from shutil import copyfile

CV_PATIENTS = (PATIENT, "F7177163C833DFF4B38FC8D2872F1EC6")
cv_root = OUTPUT_DIR / "cross_validation" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
cv_tile_root = cv_root / "tiles"
cv_series = set()
cv_yamls = {}

for patient in CV_PATIENTS:
    patient_dir = ROOT / "Data" / patient
    patient_images = sorted((patient_dir / "MS").glob("*.dcm"))
    consensus_files = sorted((patient_dir / "ANN").glob("*consensus*.dcm"))
    roi_files = sorted((patient_dir / "ANN").glob("*Monolayer*.dcm"))
    if not patient_images or len(consensus_files) != 1 or len(roi_files) != 1:
        raise ValueError(
            f"Expected a DICOM slide, one consensus file, and one ROI file in {patient_dir}."
        )

    headers = [pydicom.dcmread(p, stop_before_pixels=True) for p in patient_images]
    series = {header.SeriesInstanceUID for header in headers}
    if len(series) != 1 or series & cv_series:
        raise ValueError(f"Expected one distinct slide series for patient {patient}.")
    cv_series.update(series)
    native = next(
        header
        for header in headers
        if list(header.get("ImageType", []))[:3] == ["ORIGINAL", "PRIMARY", "VOLUME"]
    )
    for annotation_path in (consensus_files[0], roi_files[0]):
        annotation = pydicom.dcmread(annotation_path, stop_before_pixels=True)
        refs = {
            item.ReferencedSOPInstanceUID
            for item in annotation.get("ReferencedImageSequence", [])
        }
        if annotation.get("PixelOriginInterpretation") != "VOLUME" or refs != {
            native.SOPInstanceUID
        }:
            raise ValueError(
                f"{annotation_path.name} must reference this slide's level-0 pixel matrix."
            )

    patient_boxes = read_boxes(consensus_files[0], excluded_labels=("artifact",))
    patient_rois = read_boxes(roi_files[0])
    patient_rois = np.column_stack(
        (np.ceil(patient_rois[:, :2]), np.floor(patient_rois[:, 2:]))
    ).astype(int)
    roi_sizes = patient_rois[:, 2:] - patient_rois[:, :2]
    slide_dimensions = (native.TotalPixelMatrixColumns, native.TotalPixelMatrixRows)
    if (
        np.any(patient_rois[:, :2] < 0)
        or np.any(patient_rois[:, 2:] > slide_dimensions)
        or np.any(roi_sizes < TILE_SIZE)
        or np.any(roi_sizes % TILE_SIZE)
    ):
        raise ValueError(
            f"Expected in-bounds regions that can be tiled exactly for patient {patient}."
        )

    print(f"{patient}: {len(patient_boxes)} source boxes, {len(patient_rois)} regions")
    with WsiDicom.open(patient_images) as patient_slide:
        for split, stride in (("train", TRAIN_STRIDE), ("val", TILE_SIZE)):
            output = cv_tile_root / patient / split
            output.mkdir(parents=True)
            tile_count = box_count = 0
            for roi_index, (left, top, right, bottom) in enumerate(patient_rois):
                for y in tile_starts(top, bottom, stride):
                    for x in tile_starts(left, right, stride):
                        stem = output / f"roi{roi_index}_x{x}_y{y}"
                        tile_boxes = np.unique(
                            labels_for_tile(patient_boxes, x, y, TILE_SIZE), axis=0
                        )
                        tile_image = patient_slide.read_region(
                            (x, y), 0, (TILE_SIZE, TILE_SIZE)
                        )
                        tile_image.convert("RGB").save(stem.with_suffix(".png"))
                        stem.with_suffix(".txt").write_text(
                            "".join(
                                "0 " + " ".join(f"{value:.8f}" for value in box) + "\n"
                                for box in tile_boxes
                            ),
                            encoding="utf-8",
                        )
                        tile_count += 1
                        box_count += len(tile_boxes)
            if not box_count:
                raise ValueError(f"No cell labels for patient {patient} in {split}.")
            print(f"  {split}: {tile_count} tiles, {box_count} box instances")

for held_out in CV_PATIENTS:
    training_patient = next(patient for patient in CV_PATIENTS if patient != held_out)
    fold = cv_root / f"holdout_{held_out}"
    for split, patient in (("train", training_patient), ("val", held_out)):
        for folder, extension in (("images", "*.png"), ("labels", "*.txt")):
            destination = fold / folder / split
            destination.mkdir(parents=True)
            for source in (cv_tile_root / patient / split).glob(extension):
                copyfile(source, destination / source.name)

    data = fold / "data.yaml"
    data.write_text(
        yaml.safe_dump(
            {
                "path": fold.as_posix(),
                "train": "images/train",
                "val": "images/val",
                "names": {0: "cell"},
            }
        ),
        encoding="utf-8",
    )
    cv_yamls[held_out] = data

### Train and evaluate both folds

Initialize a fresh model from the same pretrained weights in each fold. Train for a fixed number of epochs and evaluate the final checkpoint on the held-out patient. The cell reports mAP50 and mAP50-95 for each fold and their unweighted means. It saves the individual scores and checkpoint paths in `fold_metrics.csv`. With two patients, the scores describe transfer between these two slides.

In [ ]:
import torch
from ultralytics import YOLO

cv_device = 0 if torch.cuda.is_available() else "cpu"
cv_results = []
for held_out, fold_yaml in cv_yamls.items():
    fold_model = YOLO(MODEL_WEIGHTS)
    fold_model.train(
        data=str(fold_yaml),
        imgsz=TILE_SIZE,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        device=cv_device,
        workers=0,
        seed=SEED,
        mosaic=0.0,
        amp=False,
        val=False,
        project=str(fold_yaml.parent),
        name="train",
    )
    checkpoint = Path(fold_model.trainer.last)
    detector = YOLO(str(checkpoint))
    scores = detector.val(
        data=str(fold_yaml),
        imgsz=TILE_SIZE,
        batch=BATCH_SIZE,
        device=cv_device,
        workers=0,
        project=str(fold_yaml.parent),
        name="held_out",
    )
    cv_results.append(
        {
            "held_out_patient": held_out,
            "map50": scores.box.map50,
            "map50_95": scores.box.map,
            "checkpoint": str(checkpoint),
        }
    )

with (cv_root / "fold_metrics.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(cv_results[0]))
    writer.writeheader()
    writer.writerows(cv_results)

for row in cv_results:
    print(
        f"Held out {row['held_out_patient']}: mAP50={row['map50']:.3f}, mAP50-95={row['map50_95']:.3f}"
    )
print(f"Mean mAP50: {np.mean([row['map50'] for row in cv_results]):.3f}")
print(f"Mean mAP50-95: {np.mean([row['map50_95'] for row in cv_results]):.3f}")
print(cv_root / "fold_metrics.csv")

## Inspect predictions on the second slide

For inference, run **Setup** and the final code cell. The defaults select the second patient's DICOM pyramid and a crop inside an annotated tissue region. The cell applies the baseline `best.pt` checkpoint trained on the first patient.

Set `ROI_X` and `ROI_Y` to the left and top coordinates of the desired crop in level-0 pixels. Set `ROI_CONFIDENCE` to the minimum prediction confidence to display. The cell reads a native-resolution 1024 by 1024 pixel crop, displays the detected boxes, and writes a CSV with box coordinates in the full-resolution slide coordinate system.

In [ ]:
import torch
from ultralytics import YOLO

ROI_PYRAMID_DIR = ROOT / "Data" / "F7177163C833DFF4B38FC8D2872F1EC6" / "MS"
ROI_X = 72989
ROI_Y = 272069
ROI_CONFIDENCE = 0.25
ROI_CHECKPOINT = OUTPUT_DIR / "runs" / "cell_boxes" / "weights" / "best.pt"

pyramid_dir = Path(ROI_PYRAMID_DIR).expanduser()
if not pyramid_dir.is_dir():
    raise NotADirectoryError(f"Expected a DICOM pyramid folder: {pyramid_dir}")
roi_paths = sorted(
    path
    for path in pyramid_dir.iterdir()
    if path.is_file() and path.suffix.lower() == ".dcm"
)
if not roi_paths:
    raise FileNotFoundError(f"No DICOM image files found in {pyramid_dir}")

training_paths = sorted(WSI_DIR.glob("*.dcm"))
if not training_paths:
    raise FileNotFoundError(
        f"Cannot verify the training slide identity. No DICOM files found in {WSI_DIR}"
    )

training_headers = [
    pydicom.dcmread(path, stop_before_pixels=True) for path in training_paths
]
roi_headers = [pydicom.dcmread(path, stop_before_pixels=True) for path in roi_paths]
training_series = {header.SeriesInstanceUID for header in training_headers}
training_instances = {header.SOPInstanceUID for header in training_headers}
roi_series = {header.SeriesInstanceUID for header in roi_headers}
if roi_series & training_series or any(
    header.SOPInstanceUID in training_instances for header in roi_headers
):
    raise ValueError(
        "Choose a different slide. This folder contains DICOM images from the training slide."
    )
if len(roi_series) != 1:
    raise ValueError(
        "Set ROI_PYRAMID_DIR to a folder containing one DICOM slide series."
    )

roi_start = np.array([ROI_X, ROI_Y])
roi_end = roi_start + TILE_SIZE
if not ROI_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"Set ROI_CHECKPOINT to a trained best.pt file. Missing: {ROI_CHECKPOINT}"
    )

with WsiDicom.open(roi_paths) as slide:
    level0 = slide.levels[0]
    roi_slide_size = (level0.size.width, level0.size.height)
    has_native_image = any(
        list(header.get("ImageType", []))[:3] == ["ORIGINAL", "PRIMARY", "VOLUME"]
        and "RESAMPLED" not in header.ImageType
        and (header.get("TotalPixelMatrixColumns"), header.get("TotalPixelMatrixRows"))
        == roi_slide_size
        for header in roi_headers
    )
    if not has_native_image:
        raise ValueError(
            "The level-0 image is missing or marked as resampled. Select the original full-resolution DICOM volume."
        )
    if np.any(roi_start < 0) or np.any(roi_end > roi_slide_size):
        raise ValueError(
            f"The ROI must fit inside the full-resolution slide of size {roi_slide_size}."
        )
    roi_image = slide.read_region((ROI_X, ROI_Y), 0, (TILE_SIZE, TILE_SIZE)).convert(
        "RGB"
    )

print(f"Level-0 slide size: {roi_slide_size[0]} x {roi_slide_size[1]} pixels")
roi_detector = YOLO(str(ROI_CHECKPOINT))
roi_result = roi_detector.predict(
    source=roi_image,
    imgsz=TILE_SIZE,
    conf=ROI_CONFIDENCE,
    device=0 if torch.cuda.is_available() else "cpu",
    verbose=False,
)[0]
roi_overlay = Image.fromarray(roi_result.plot(labels=False)[:, :, ::-1])

roi_output = (
    OUTPUT_DIR
    / "inference"
    / (f"{next(iter(roi_series))}_x{ROI_X}_y{ROI_Y}_{datetime.now():%Y%m%d-%H%M%S-%f}")
)
roi_output.mkdir(parents=True)
roi_image.save(roi_output / "roi.png")
roi_overlay.save(roi_output / "predicted_boxes.png")
with (roi_output / "predicted_boxes.csv").open(
    "w", newline="", encoding="utf-8"
) as handle:
    writer = csv.writer(handle)
    writer.writerow(
        ["x_min_level0", "y_min_level0", "x_max_level0", "y_max_level0", "confidence"]
    )
    offset = np.tile(roi_start, 2)
    for box, confidence in zip(
        roi_result.boxes.xyxy.cpu().numpy(),
        roi_result.boxes.conf.cpu().numpy(),
        strict=True,
    ):
        writer.writerow([*(float(value) for value in box + offset), float(confidence)])

print(f"{len(roi_result.boxes)} predicted boxes at confidence >= {ROI_CONFIDENCE:.2f}")
print(f"ROI origin: ({ROI_X}, {ROI_Y}). Saved to {roi_output}")
display(roi_overlay)